In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def get_group(group):
    return 1 if len(group) > 1 else 0

df = pd.read_csv('../data/train.csv', encoding="latin8")

df['cabin_prefix'] = df['Cabin'].apply(lambda x: str(x)[0])
deck_counts = df['cabin_prefix'].value_counts()
df['Deck_Frequency'] = df['cabin_prefix'].map(deck_counts)
# df['Has_Cabin'] = df['Cabin'].notna().astype(int)
df['family_size'] = df['SibSp'] + df['Parch']
df['Sex'] = df['Sex'].map({"male": 0, "female": 1})
df['Embarked'] = df['Embarked'].map({'S': 3, 'Q': 1, 'C': 2})
df.fillna({'Embarked': 0}, inplace=True)
df['dollar_age'] = df['Fare']/df['Age']
df['in_group'] = df.groupby('Ticket')['Ticket'].transform(lambda x: 1 if len(x) > 1 else 0)
df.fillna({'dollar_age': df['dollar_age'].mean()}, inplace=True)
df

In [ ]:
age = df['in_group'].to_list()
fare = df['Sex'].to_list()
survived = df['Survived'].to_list()
plt.scatter(x=age, y=fare, c=survived, alpha=0.5)
plt.legend()
plt.show()

In [ ]:
df['Cabin'].value_counts()

In [ ]:
import math

class ID3:
    def __init__(self, df, target_attribute: str, attributes):
        self.df = df
        self.target_attribute = target_attribute
        self.attributes = attributes
        self.root = None
    
    def get_target_entropy(self, df: pd.DataFrame, target_atr: str)-> float:
        """
        Gets the total entropy of the dataset by summing entropy for 
        each possible value of the target concept

        param df: the entire dataframe
        param target_atr: the target concept to be examined
        returns float. The total entropy for the target_atr
        """
        att_list = df[target_atr].unique() # All possible c's that S can take
        return sum([self.calculate_entropy(df, target_atr, atr) for atr in att_list])

    
    def calculate_entropy(self, df: pd.DataFrame, target_atr: str, atr: str) -> float:
        """
        Calculates single class entropy based on the passed target_atr and atr to get the 
        entropy based on the possible value of the target_atr

        param df: the entire dataframe
        param target_atr: the column that is the target concept
        param atr: a possible value that the target_atr can take

        returns float. The entropy of the target_atr
        """
        atr_df = df[df[target_atr] == atr]
        num = len(atr_df)
        denom = len(df)

        p_pos = num / denom # pi 
        return -1 * p_pos * math.log2(p_pos) #entropy

    def information_gain(self, df: pd.DataFrame, new_attr: str, target_atr: str) -> float:
        """
        Calculates information gain based on a target attribute passed in

        param df: the entire dataframe
        param new_attr: the attribute that is going to be used to measure against the target_attr
        param target_atr: the target concept

        returns float. The information gain based on the new_attr
        """
        target_ent = self.get_target_entropy(df, target_atr) #Severity entropy: 0.89
        new_attr_values = df[new_attr].unique() # All possible values new_attr can take
        temp = 0

        for attr in new_attr_values:
            slim_df = df[df[new_attr] == attr] # Sv

            for attr_2 in slim_df[target_atr].unique():
                # Sv/S: len(slim_df)/len(df)
                temp += (len(slim_df)/len(df)) * self.calculate_entropy(slim_df, target_atr, attr_2)
        
        return target_ent - temp
    
    def split_information(self, df: pd.DataFrame, attr: str) -> float:
        """
        Split Information (Intrinsic Value) of attribute `attr` on dataset `df`.
        - Uses p_v * log2(p_v) terms.
        - Treats NaN as its own category (set dropna=True to ignore NaNs).
        """
        n = len(df)
        if n == 0:
            return 0.0  # or raise ValueError("Empty dataframe")

        # p_v for each distinct value (including NaN as a category)
        p = df[attr].value_counts(normalize=True, dropna=False)

        # SplitInfo = - sum p * log2 p  (skip p==0 just in case)
        split_info = -sum(pi * math.log2(pi) for pi in p.values if pi > 0)
        return float(split_info)


    def gain_ratio(self, df: pd.DataFrame, target_attr: str, new_attr: str) -> float:
        """
        Gain Ratio = Information Gain / Split Information.
        - Returns 0 if SplitInfo == 0 to avoid division-by-zero.
        - Optionally clamp tiny negative IGs to 0.
        """
        ig = self.information_gain(df, new_attr, target_attr)  # ensure signature matches
        # Numerical safety: IG should be >= 0 theoretically
        if ig < 0:
            ig = 0.0

        si = self.split_information(df, new_attr)
        if si == 0:
            return 0.0
        return ig / si
    

id3 = ID3(df, 'Survived', ['Ticket', 'Cabin', 'Sex'])
id3.gain_ratio(df, "Survived", "Cabin")

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler, TargetEncoder
from sklearn.model_selection import KFold, GridSearchCV, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

# df = pd.read_csv('../data/train.csv', encoding="latin8")

X = df.drop(columns=['cabin_prefix', 'Survived', 'PassengerId', "Name", "Cabin", "Ticket", 'dollar_age'])
y = df['Survived']
X.fillna({"Deck_Frequency": 0, 'Age': X['Age'].mean()}, inplace=True)
X

In [ ]:
X.isna().sum()

In [96]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
cols = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'Deck_Frequency', 'family_size']
X = df[cols]
X.fillna({'Age': df['Age'].mean()}, inplace=True)
# X = df.drop(columns=['Survived', 'PassengerId', "Name", "Cabin", "Ticket", 'dollar_age'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
param_grid = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'splitter': ['best', 'random'],
    'max_depth': [None, 5, 10, 15, 20, 25, 30],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': [None, 'sqrt', 'log2'],
    'max_leaf_nodes': [None, 10, 20, 50, 100],
    'min_impurity_decrease': [0.0, 0.01, 0.05, 0.1],
    'class_weight': [None, 'balanced']
}

params =  {'class_weight': None, 
        'criterion': 'gini', 
        'max_depth': 10, 
        'max_features': None, 
        'max_leaf_nodes': 100, 
        'min_impurity_decrease': 0.0, 
        'min_samples_leaf': 2, 
        'min_samples_split': 2, 
        'splitter': 'random'}

tree = DecisionTreeClassifier()
grid = GridSearchCV(
    estimator=tree, param_grid=param_grid, verbose=2
)
grid.fit(X_train, y_train)
y_pred = grid.predict(X_test)
print(accuracy_score(y_true = y_test, y_pred=y_pred))
print(confusion_matrix(y_true = y_test, y_pred=y_pred))
# print(f"Best parameters: {grid.best_params_}")
# print(f"Best score: {grid.best_score_:.4f}")



In [ ]:
X.isna().sum()

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=10),  
    n_estimators=1000,
    learning_rate=1.2,
    random_state=42
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred))

param_grid = {
    'n_estimators': [30, 50, 75, 100, 1000, 2000],
    'learning_rate': [0.8, 1.0, 1.2, 1.5, 2]
}

ada_model = AdaBoostClassifier(random_state=42)
grid_search = GridSearchCV(ada_model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"Best params: {grid_search.best_params_}")
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_pred=y_pred, y_true=y_test))


In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
# 20 minute run time
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 0.001, 0.01, 0.1, 1],
    'kernel': ['rbf', 'linear', 'poly']
}

svm = SVC(random_state=42)
grid = GridSearchCV(svm, param_grid, cv=5, scoring='accuracy')
grid.fit(X_train_scaled, y_train)

print(f"Best params: {grid.best_params_}")
y_pred = grid.best_estimator_.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")